# 2 - Calculate metrics (template)

This template runs trajectory metrics and builds feeder/exit visit tables. Adjust config/time_division as needed.

In [ ]:
from pathlib import Path
import glob
import pandas as pd

import bb_metrics
cfg = bb_metrics.load_config('/path/to/your/season_config.py')  # replace with your config
from bb_metrics import metrics_pipeline as mp
from bb_metrics import metricsfunctions as mfunc  # uses cfg definitions
import bb_metrics.datafunctions as dfunc

bb_metrics.set_config(cfg)

## Trajectory metrics (per time division)
- Builds paired files by hive/start/end
- Runs `datafile_to_metrics` in parallel
- Outputs to `cfg.metrics_dir`

In [ ]:
# Collect trajectory parquet files
trajdir = cfg.traj_outdir
all_datafiles = sorted(Path(trajdir).glob("*.parquet"))

# Build pairs (one per hive + time window)
pairs, unmatched = mp.build_pairs_from_traj(all_datafiles, cfg.cam_hive_map)
print(f"pairs: {len(pairs)}, unmatched: {len(unmatched)}")

# Parameters
metrics_dir = cfg.metrics_dir
reprocess = False   # force re-processing of all metrics
update = True      # only recompute segments where source data changed
time_division = "60min"  # "1min", "5min", or "60min"
min_num_detections = 90
save_xy_hist = True
num_processes = 2


# Optional: combhist via annotation grids
use_comb = True
grid_lookup = None
if use_comb:
    ds = 8
    grid_dir = cfg.comb_images_root / "annotation_grids"
    ann_df = pd.read_parquet(grid_dir / f"annotation_grids_ds_{ds}.parquet")
    ann_df["grid_path"] = ann_df["grid_path"].astype(str)
    grid_lookup = dfunc.GridLookup(ann_df)

# Run metrics (combhists are only calculated if grid_lookup is passed in)
mp.run_metrics_from_pairs(
    pairs,
    reprocess=reprocess,
    update=update,
    time_division=time_division,
    min_num_detections=min_num_detections,
    save_xy_hist=save_xy_hist,
    metrics_dir=metrics_dir,
    num_processes=num_processes,
    grid_lookup=grid_lookup,
)


## Feeder/exit visit tables
- Uses daily feeder/exit detection files (clahe/non-clahe) from `cfg.feedercam_daily_dir`
- Builds visit tables and saves to `cfg.metrics_dir`

In [ ]:
datadir = cfg.feedercam_daily_dir
metrics_dir = cfg.metrics_dir

feedercam_files = mp.pair_by_date(datadir, "2025*feedercam-c.parquet", "2025*feedercam-nc.parquet")
exitcam_files   = mp.pair_by_date(datadir, "2025*exitcam-c.parquet",   "2025*exitcam-nc.parquet")

visit_gap_seconds = 15
confidence_threshold = 0.0

df_feedervisits = mp.process_visit_pairs(
    feedercam_files,
    visit_gap_seconds=visit_gap_seconds,
    confidence_threshold=confidence_threshold,
)
df_exitvisits = mp.process_visit_pairs(
    exitcam_files,
    visit_gap_seconds=visit_gap_seconds,
    confidence_threshold=confidence_threshold,
)

metrics_dir = Path(metrics_dir)
metrics_dir.mkdir(parents=True, exist_ok=True)
df_feedervisits.to_parquet(metrics_dir / 'df_feedervisits.parquet')
df_exitvisits.to_parquet(metrics_dir / 'df_exitvisits.parquet')
df_feedervisits.head(), df_exitvisits.head()


## Day metrics / histograms / death estimation
These downstream analyses can reuse the outputs above (per-time metrics, visit tables). Keep or adapt your existing analysis code here.

### Day data matrix
Aggregate per-time metrics into per-day metrics, then merge feeder/exit visit summaries.


In [ ]:
metrics_dir = cfg.metrics_dir
df_metrics = mp.load_metrics_files(metrics_dir, pattern=f"metrics-{time_division}*")

daysine_peaktime = 13.5  # local time (Europe/Berlin) for peak sunlight
sumqs = ["num_detections", "num_trips", "inplace_events", "burst_events", "large_turn_events"]

dfday = mp.build_day_data_matrix(
    df_metrics,
    cfg.day_to_number,
    tz="Europe/Berlin",
    sumqs=sumqs,
    daysine_peaktime=daysine_peaktime,
)

df_feedervisits, df_exitvisits = mp.load_visit_tables(metrics_dir)
dfday = mp.merge_daily_visit_metrics(dfday, df_feedervisits, df_exitvisits, cfg.day_to_number)

output_path = mp.save_day_data_matrix(dfday, metrics_dir)
print(f"Wrote dfday to file: {output_path}")
dfday.head()


### Day histograms
Combine hourly xyhist files into per-day histograms (HDF5).


In [ ]:
xyhist_dir = metrics_dir
output_hdf5_file = mp.build_day_xyhist(
    xyhist_dir,
    output_hdf5_file=metrics_dir / "dayxyhist.h5",
    tz="Europe/Berlin",
)
output_hdf5_file


### Death estimation (changepoint model)
Runs a PyMC changepoint model to estimate the day of death per bee.


In [ ]:
dfday = pd.read_csv(metrics_dir / "daydatamat.csv")

estimator = mp.LifetimeEstimator()
df_beedeath = mp.estimate_death_days(
    dfday,
    estimator=estimator,
    progress=False,
)

df_beedeath.to_csv(metrics_dir / "df_beedeath.csv", index=False)
df_beedeath.head()
